# Traffic Lens — Run Everything (Colab)

**Runtime > Change runtime type > T4 GPU**, then Run all. This one notebook does
every piece of Project 2's hands-on work and produces every artifact the repo's
README needs: a calibrated detect/track/count/speed run, a ByteTrack parameter
sweep, an auto-labeling demo, a VisDrone fine-tune with a real mAP comparison, and
a functional test of the upload-a-video app. Everything downloads at the end —
nothing needs to run on a local machine.

## 0. Setup

In [ ]:
!pip install -q ultralytics "supervision>=0.29,<0.30" transformers opencv-python-headless "numpy<2.4"

In [ ]:
import json
import shutil
import zipfile
from collections import defaultdict, deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from ultralytics import YOLO

from google.colab import files

WORK = Path("/content/traffic_lens_run")
WORK.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Get the demo video, look at a real frame

Calibration and line placement need to match what's actually in the footage — so we
look at a frame before deciding either, instead of guessing blind.

In [ ]:
from supervision.assets import VideoAssets, download_assets

video_path = download_assets(VideoAssets.VEHICLES)
info = sv.VideoInfo.from_video_path(video_path)
print(f"video: {info.width}x{info.height} @ {info.fps}fps, {info.total_frames} frames")

first_frame = next(sv.get_video_frames_generator(video_path))
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.title("frame 0 — use this to place the counting line and calibration points")
plt.axis("off")
plt.show()

## 2. Core pipeline (same logic as `track_count_speed.py`, copied inline so this
notebook is fully self-contained and doesn't depend on repo file versions)

In [ ]:
VEHICLE_CLASSES = [2, 3, 5, 7]  # COCO: car, motorcycle, bus, truck


def load_model(name="yolo26n.pt"):
    try:
        return YOLO(name)
    except Exception as e:
        print(f"could not load {name} ({e}); falling back to yolo11n.pt")
        return YOLO("yolo11n.pt")


class ViewTransformer:
    def __init__(self, source_px, target_m):
        self.m, _ = cv2.findHomography(np.array(source_px, dtype=np.float32),
                                       np.array(target_m, dtype=np.float32))

    def transform(self, points):
        if len(points) == 0:
            return points
        pts = points.reshape(-1, 1, 2).astype(np.float32)
        return cv2.perspectiveTransform(pts, self.m).reshape(-1, 2)


def process_video(source, output_path, model=None, model_name="yolo26n.pt", conf=0.3,
                  max_frames=0, calib=None, line_y_frac=0.5,
                  track_activation_threshold=0.25, lost_track_buffer=30,
                  progress_every=25, verbose=True):
    if model is None:
        model = load_model(model_name)
    info = sv.VideoInfo.from_video_path(source)
    fps = info.fps

    tracker = sv.ByteTrack(frame_rate=fps,
                           track_activation_threshold=track_activation_threshold,
                           lost_track_buffer=lost_track_buffer)
    line_y = int(info.height * line_y_frac)
    line = sv.LineZone(start=sv.Point(0, line_y), end=sv.Point(info.width, line_y))

    box_ann, label_ann = sv.BoxAnnotator(thickness=2), sv.LabelAnnotator(text_scale=0.5)
    trace_ann, line_ann = sv.TraceAnnotator(trace_length=int(fps)), sv.LineZoneAnnotator(text_scale=0.6)

    view = ViewTransformer(calib["source_px"], calib["target_m"]) if calib else None
    history = defaultdict(lambda: deque(maxlen=int(fps)))
    seen_ids = set()

    with sv.VideoSink(output_path, video_info=info) as sink:
        for idx, frame in enumerate(sv.get_video_frames_generator(source)):
            if max_frames and idx >= max_frames:
                break
            result = model(frame, conf=conf, verbose=False)[0]
            det = sv.Detections.from_ultralytics(result)
            det = det[np.isin(det.class_id, VEHICLE_CLASSES)]
            det = tracker.update_with_detections(det)
            line.trigger(det)
            seen_ids.update(det.tracker_id.tolist())

            labels = []
            for tid, anchor in zip(det.tracker_id, det.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)):
                speed_txt = ""
                if view is not None:
                    y_m = view.transform(anchor.reshape(1, 2))[0][1]
                    history[tid].append((y_m, idx))
                    if len(history[tid]) >= fps // 2:
                        (y0, f0), (y1, f1) = history[tid][0], history[tid][-1]
                        mps = abs(y1 - y0) / ((f1 - f0) / fps)
                        speed_txt = f" {mps * 3.6:.0f} km/h"
                labels.append(f"#{tid}{speed_txt}")

            frame = trace_ann.annotate(frame, det)
            frame = box_ann.annotate(frame, det)
            frame = label_ann.annotate(frame, det, labels=labels)
            frame = line_ann.annotate(frame, line_counter=line)
            sink.write_frame(frame)
            if verbose and idx % progress_every == 0:
                print(f"frame {idx}: {len(det)} vehicles  (in={line.in_count} out={line.out_count})")

    return {"in_count": line.in_count, "out_count": line.out_count,
            "total_unique_ids": len(seen_ids), "output_path": output_path,
            "has_speed": view is not None}


shared_model = load_model("yolo26n.pt")
print("model ready")

## 3. Calibration

Pick 4 points in the frame above that form a real-world rectangle (e.g. lane-marking
corners), and the real-world size of that rectangle in meters. **Look at the frame
printed in step 1 and adjust `source_px` below to match what you actually see** —
these starting values assume a roughly centered two-lane view; they're a starting
point, not a measurement.

In [ ]:
calib_demo = {
    "source_px": [[int(info.width * 0.35), int(info.height * 0.55)],
                  [int(info.width * 0.65), int(info.height * 0.55)],
                  [int(info.width * 0.85), int(info.height * 0.95)],
                  [int(info.width * 0.15), int(info.height * 0.95)]],
    "target_m": [[0, 0], [7, 0], [7, 30], [0, 30]],  # ~2 lanes wide (7m), 30m of road
}

preview = first_frame.copy()
pts = np.array(calib_demo["source_px"], dtype=np.int32)
cv2.polylines(preview, [pts], isClosed=True, color=(0, 0, 255), thickness=3)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title("calibration quad overlaid — adjust source_px above if this doesn't match real lane geometry")
plt.axis("off")
plt.show()

(WORK / "calib_demo.json").write_text(json.dumps(calib_demo, indent=2))
print("saved calib_demo.json -- this is an ESTIMATED calibration for demo purposes, not a verified real-world measurement")

## 4. Calibrated run with real line placement

Pick `line_y_frac` to match a sensible crossing point in the frame (default 0.6 puts
it a bit below center, roughly where the calibration quad sits) rather than the
scaffold's arbitrary frame-middle default.

In [ ]:
stats_calibrated = process_video(
    source=video_path, output_path=str(WORK / "annotated_calibrated.mp4"),
    model=shared_model, calib=calib_demo, line_y_frac=0.6, max_frames=200,
)
print(stats_calibrated)

## 5. ByteTrack parameter sweep

`total_unique_ids` is the proxy for tracking quality here: for a roughly constant
number of real vehicles in frame, a setting that fragments tracks into more IDs than
another is doing worse at maintaining identity through occlusion/misses.

In [ ]:
sweep_settings = [
    {"track_activation_threshold": 0.5, "lost_track_buffer": 10},
    {"track_activation_threshold": 0.25, "lost_track_buffer": 30},
    {"track_activation_threshold": 0.1, "lost_track_buffer": 60},
]
sweep_results = []
for cfg in sweep_settings:
    stats = process_video(
        source=video_path, output_path=str(WORK / f"sweep_{cfg['track_activation_threshold']}.mp4"),
        model=shared_model, max_frames=200, verbose=False, **cfg,
    )
    row = {**cfg, "total_unique_ids": stats["total_unique_ids"],
           "in_count": stats["in_count"], "out_count": stats["out_count"]}
    sweep_results.append(row)
    print(row)

(WORK / "bytetrack_sweep.json").write_text(json.dumps(sweep_results, indent=2))

## 6. Auto-labeling demo (Grounding DINO)

Extracts a handful of frames and produces YOLO-format labels + preview images from a
text prompt alone — no training. Ready to import into CVAT for correction afterward
(that correction step, and timing it against labeling from scratch, is worth doing by
hand later — it's not something to automate away).

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

GD_MODEL = "IDEA-Research/grounding-dino-tiny"
gd_processor = AutoProcessor.from_pretrained(GD_MODEL)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD_MODEL).to(device)

frames_dir = WORK / "autolabel_frames"
(frames_dir / "images").mkdir(parents=True, exist_ok=True)
(frames_dir / "labels").mkdir(exist_ok=True)
(frames_dir / "preview").mkdir(exist_ok=True)

classes = ["car", "truck", "bus", "motorcycle"]
text_prompt = ". ".join(classes) + "."

extracted = []
for i, frame in enumerate(sv.get_video_frames_generator(video_path)):
    if i % 40 == 0 and len(extracted) < 6:
        p = frames_dir / "images" / f"frame_{i:04d}.jpg"
        cv2.imwrite(str(p), frame)
        extracted.append(p)
    if len(extracted) >= 6:
        break

from PIL import Image, ImageDraw

total_boxes = 0
for path in extracted:
    image = Image.open(path).convert("RGB")
    inputs = gd_processor(images=image, text=text_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gd_model(**inputs)
    result = gd_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=0.35, text_threshold=0.35,
        target_sizes=[image.size[::-1]])[0]

    w, h = image.size
    lines, draw_img = [], image.copy()
    draw = ImageDraw.Draw(draw_img)
    for box, label in zip(result["boxes"], result["text_labels"]):
        cls = next((i for i, c in enumerate(classes) if c in label), None)
        if cls is None:
            continue
        x0, y0, x1, y1 = box.tolist()
        lines.append(f"{cls} {(x0+x1)/2/w:.6f} {(y0+y1)/2/h:.6f} {(x1-x0)/w:.6f} {(y1-y0)/h:.6f}")
        draw.rectangle([x0, y0, x1, y1], outline="red", width=3)
        draw.text((x0, max(0, y0 - 12)), label, fill="red")

    (frames_dir / "labels" / f"{path.stem}.txt").write_text("\n".join(lines))
    draw_img.save(frames_dir / "preview" / path.name)
    total_boxes += len(lines)

print(f"{total_boxes} boxes across {len(extracted)} frames")
plt.figure(figsize=(14, 6))
for i, p in enumerate((frames_dir / "preview").glob("*.jpg")):
    plt.subplot(2, 3, i + 1)
    plt.imshow(Image.open(p))
    plt.axis("off")
plt.suptitle("auto-labeled preview — ready for CVAT import + correction")
plt.tight_layout()
plt.show()

shutil.make_archive(str(WORK / "autolabel_demo"), "zip", frames_dir)

## 7. VisDrone fine-tune + real mAP comparison

Sanity-checks the training call on Ultralytics' tiny bundled `coco8` set first (fast,
catches config errors) before committing to the full VisDrone run, which auto-
downloads a properly-labeled public drone-traffic dataset — no manual correction
needed for this part. **This step is the long one** (VisDrone download + ~15-20
epochs on a T4 is roughly 30-60+ minutes) — if it stalls or you hit a Colab session
limit, that's the signal to move to the remote VM fallback instead of forcing it.

In [ ]:
sanity = YOLO("yolo26n.pt").train(data="coco8.yaml", epochs=1, imgsz=640, verbose=False)
print("sanity check passed -- train call works, proceeding to VisDrone")

In [ ]:
finetune_results = YOLO("yolo26n.pt").train(
    data="VisDrone.yaml", epochs=20, imgsz=640, project=str(WORK), name="visdrone_finetune",
)
best_weights = WORK / "visdrone_finetune" / "weights" / "best.pt"
print("fine-tuned weights:", best_weights)

In [ ]:
print("== pretrained COCO YOLO26n on VisDrone val ==")
metrics_pretrained = YOLO("yolo26n.pt").val(data="VisDrone.yaml")
print(f"mAP50-95: {metrics_pretrained.box.map:.4f}  mAP50: {metrics_pretrained.box.map50:.4f}")

print("\n== fine-tuned YOLO26n on VisDrone val ==")
metrics_finetuned = YOLO(str(best_weights)).val(data="VisDrone.yaml")
print(f"mAP50-95: {metrics_finetuned.box.map:.4f}  mAP50: {metrics_finetuned.box.map50:.4f}")

finetune_summary = {
    "pretrained_map50_95": float(metrics_pretrained.box.map),
    "pretrained_map50": float(metrics_pretrained.box.map50),
    "finetuned_map50_95": float(metrics_finetuned.box.map),
    "finetuned_map50": float(metrics_finetuned.box.map50),
    "epochs": 20,
}
(WORK / "visdrone_finetune_summary.json").write_text(json.dumps(finetune_summary, indent=2))
shutil.copy(best_weights, WORK / "yolo26n_visdrone_finetuned.pt")
print(finetune_summary)

## 8. App function test

`app.py`'s Gradio callback is a thin wrapper around this exact `process_video` call --
testing it here proves the app works without needing a live server. (To actually poke
at the UI, `python app.py` launches it -- that's just using the tool, not verifying it.)

In [ ]:
app_test_stats = process_video(
    source=video_path, output_path=str(WORK / "app_test_output.mp4"),
    model=shared_model, calib=calib_demo, line_y_frac=0.6, max_frames=120, verbose=False,
)
print("app function test:", app_test_stats)
assert Path(app_test_stats["output_path"]).stat().st_size > 0, "app produced an empty video!"
print("app.py's core function works end-to-end.")

## 9. Download everything

One artifact per cell so a single failed download doesn't block the rest.

In [ ]:
files.download(str(WORK / "calib_demo.json"))
files.download(str(WORK / "annotated_calibrated.mp4"))
files.download(str(WORK / "bytetrack_sweep.json"))
files.download(str(WORK / "autolabel_demo.zip"))
files.download(str(WORK / "visdrone_finetune_summary.json"))
files.download(str(WORK / "yolo26n_visdrone_finetuned.pt"))
files.download(str(WORK / "app_test_output.mp4"))
print("all artifacts downloaded -- hand these + the printed numbers above back for the README update.")